# 🛡️ Deepfake Detection Using Vision Transformers — Full Pipeline
## A Comparative Study of Supervised and Self-Supervised Pretraining

**B.Tech Major Project · 2024-25**

---

This notebook orchestrates the **complete deepfake detection pipeline** end-to-end on Kaggle:

| Step | Script / Module | Description |
|------|----------------|-------------|
| 0 | Environment | GPU diagnostics, quota notes, install + clone |
| 1 | `extract_faces.py` | MTCNN face detection → 224×224 crops |
| 2 | `make_splits.py` | Video-level stratified train/val/test CSVs |
| 3 | `cache_embeddings.py` | Frozen backbone → per-video `.npy` embeddings |
| 4 | `train.py` | 6 model configs trained sequentially |
| 5 | `evaluate.py` | Per-run accuracy, AUC, F1, confusion matrices |
| 6 | `compare_runs.py` | Consolidated comparison table (CSV + Markdown) |
| 7 | Download | Zip `outputs/` for local download |

**Backbones**: ViT-B/16, DeiT-S/16, DINOv2-S/14 (frozen), DINOv2-S/14 (partial), EfficientNet-B0, Video Temporal Head

---
## 0 · Environment Setup

### 0.1 · GPU Information & Kaggle Quota

In [3]:
import os, subprocess, datetime, shutil

print("=" * 70)
print("  GPU & Runtime Diagnostics")
print("=" * 70)

# GPU Info
try:
    gpu_info = subprocess.check_output(["nvidia-smi"], text=True)
    print(gpu_info)
except FileNotFoundError:
    print("WARNING: No GPU detected! This notebook requires a GPU accelerator.")
    print("Go to Settings -> Accelerator -> GPU T4 x2 or P100")

# PyTorch GPU check
import torch
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device       : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version     : {torch.version.cuda}")

# Kaggle environment check
is_kaggle = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print(f"\nRunning on Kaggle : {is_kaggle}")
print(f"Kernel type      : {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'N/A')}")
print(f"Timestamp (UTC)  : {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')}")

# Quota notes
print("\n" + "=" * 70)
print("  Kaggle GPU Quota Reminder")
print("=" * 70)
print("""
  - Kaggle provides ~30 hours/week of GPU (T4) or ~20 hours/week (P100).
  - Each session auto-terminates after 12 hours (9 hours for GPU).
  - This full pipeline (6 training runs + evaluation) takes approximately:
      * T4 x2  : ~3-5 hours total (depending on dataset size)
      * P100    : ~2-4 hours total
  - Monitor your quota at: https://www.kaggle.com/me/account
  - TIP: If you are low on quota, comment out some training configs
    in Cell 4.2 and train only the models you need.
""")


  GPU & Runtime Diagnostics
Tue Oct 27 02:43:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------

/tmp/ipykernel_49/2019758490.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"Timestamp (UTC)  : {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')}")


### 0.2 · Install Dependencies & Clone Repository

In [ ]:
%%time
# ── Install requirements ──────────────────────────────────────────
!pip install -q facenet-pytorch --no-deps
!pip install -q timm albumentations grad-cam tabulate seaborn pyyaml

# ── Clone the project repository ─────────────────────────────────
REPO_URL = 'https://github.com/mamunur77/Deepfake.git'
PROJECT_DIR = '/kaggle/working/deepfake-vit'

if os.path.exists(PROJECT_DIR):
    print(f'[setup] Project directory already exists: {PROJECT_DIR}')
    print('[setup] Pulling latest changes...')
    !cd {PROJECT_DIR} && git pull origin main
else:
    print(f'[setup] Cloning repository...')
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f'\n[setup] Working directory: {os.getcwd()}')
!ls -la


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done


In [ ]:
import os

ff_path = '/kaggle/input/datasets/greatgamedota/faceforensics/cropped_images'
celeb_path = '/kaggle/input/datasets/reubensuju/celeb-df-v2'

print("--- FaceForensics Sample Folders ---")
if os.path.exists(ff_path):
    print(os.listdir(ff_path)[:15])
else:
    print("Path not found:", ff_path)

print("\n--- Celeb-DF Folders ---")
if os.path.exists(celeb_path):
    print(os.listdir(celeb_path))
else:
    print("Path not found:", celeb_path)


In [ ]:
ff_folders = os.listdir('/kaggle/input/datasets/greatgamedota/faceforensics/cropped_images')
single_folders = [f for f in ff_folders if '_' not in f]
pair_folders = [f for f in ff_folders if '_' in f]
print(f"FaceForensics single folders (real): {len(single_folders)}")
print(f"FaceForensics pair folders (fake)   : {len(pair_folders)}")


In [ ]:
import os
celeb_path = '/kaggle/input/datasets/reubensuju/celeb-df-v2'
print("Real files sample:", os.listdir(os.path.join(celeb_path, 'Celeb-real'))[:5])
print("Fake files sample:", os.listdir(os.path.join(celeb_path, 'Celeb-synthesis'))[:5])


### 0.3 · Configure Kaggle Dataset Paths

**Important**: Add your dataset as a Kaggle dataset input. Expected structure:

```
/kaggle/input/<your-dataset-name>/
├── real/          # real videos or images
└── fake/          # fake/manipulated videos or images
```

Adjust `KAGGLE_DATASET_DIR` below to match your attached dataset.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CONFIGURE DATASET PATHS (Celeb-DF v2)
# ══════════════════════════════════════════════════════════════════
import os, shutil

CELEB_DIR = "/kaggle/input/datasets/reubensuju/celeb-df-v2"
RAW_DATA_DIR = os.path.join(os.getcwd(), "data", "raw")
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# Path mapping for Celeb-DF
DATA_SOURCES = {
    "real": os.path.join(CELEB_DIR, "Celeb-real"),
    "fake": os.path.join(CELEB_DIR, "Celeb-synthesis")
}

# Link real and fake videos into data/raw/
for label, src in DATA_SOURCES.items():
    dst = os.path.join(RAW_DATA_DIR, label)
    if os.path.exists(dst) or os.path.islink(dst):
        os.remove(dst) if os.path.islink(dst) else shutil.rmtree(dst)
    
    if os.path.exists(src):
        os.symlink(src, dst)
        count = len(os.listdir(src))
        print(f"[data] Linked {label:<5} -> {src} ({count} videos)")
    else:
        print(f"[data] WARNING: Could not find {src}")

print(f"\n[data] Verified raw data directory:")
!ls -la data/raw/


In [ ]:
# # ══════════════════════════════════════════════════════════════════
# #  USER CONFIG — Set your Kaggle dataset path here
# # ══════════════════════════════════════════════════════════════════

# # Option A: FaceForensics++ dataset
# KAGGLE_DATASET_DIR = "/kaggle/input/faceforensics"  # <-- CHANGE THIS

# # Option B: Celeb-DF v2 (for cross-dataset evaluation)
# KAGGLE_CELEBDF_DIR = "/kaggle/input/celeb-df-v2"    # <-- CHANGE THIS (or set to None)

# # ── Symlink Kaggle data into the project's data/raw/ directory ──
# RAW_DATA_DIR = os.path.join(PROJECT_DIR, "data", "raw")
# os.makedirs(RAW_DATA_DIR, exist_ok=True)

# # Create symlinks for real/ and fake/ subdirectories
# for label in ["real", "fake"]:
#     src = os.path.join(KAGGLE_DATASET_DIR, label)
#     dst = os.path.join(RAW_DATA_DIR, label)
#     if os.path.exists(src):
#         if os.path.islink(dst) or os.path.exists(dst):
#             os.remove(dst) if os.path.islink(dst) else shutil.rmtree(dst)
#         os.symlink(src, dst)
#         n_files = len(list(os.scandir(src)))
#         print(f"[data] Linked: {src} -> {dst}  ({n_files} items)")
#     else:
#         print(f"[data] WARNING: Source directory not found: {src}")
#         print(f"       Please check KAGGLE_DATASET_DIR and ensure real/ and fake/ exist.")

# print(f"\n[data] Raw data directory contents:")
# !ls -la {RAW_DATA_DIR}

---
## 1 · Face Extraction (MTCNN)

Extract the largest face from each video frame with 20% margin, resize to 224×224, skip items already processed.

In [ ]:
!pip install -q facenet-pytorch --no-deps
!pip install -q timm albumentations grad-cam tabulate seaborn

!pip install -q timm albumentations facenet-pytorch grad-cam tabulate seaborn




In [ ]:
%%time
!python scripts/extract_faces.py \
    --input  data/raw \
    --output data/processed \
    --frames 16 \
    --margin 20 \
    --size   224

# Verify extraction output
import glob
real_faces = glob.glob("data/processed/real/**/*.jpg", recursive=True)
fake_faces = glob.glob("data/processed/fake/**/*.jpg", recursive=True)
print(f"\n[extract] Face crops generated:")
print(f"  Real faces : {len(real_faces):,}")
print(f"  Fake faces : {len(fake_faces):,}")
print(f"  Total      : {len(real_faces) + len(fake_faces):,}")

---
## 2 · Build Train / Val / Test Splits

Video-level stratified splitting (70/15/15). No video ID appears in more than one split.

In [ ]:
%%time
import sys
sys.path.insert(0, ".")

from src.data.make_splits import create_and_save_splits
import pandas as pd

create_and_save_splits(
    data_dir="data/processed",
    output_dir="data/splits",
    split_ratios=(0.70, 0.15, 0.15),
    seed=42,
)

# Verify splits
print("\n" + "=" * 60)
print("  Split Statistics")
print("=" * 60)
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"data/splits/{split}.csv")
    n_real = (df["label"] == 0).sum()
    n_fake = (df["label"] == 1).sum()
    n_vids = df["video_id"].nunique()
    print(f"  {split:>5}: {len(df):>6,} frames | {n_vids:>4} videos | Real: {n_real:>5,} | Fake: {n_fake:>5,}")


### 2.1 · Verify No Video Leakage Across Splits

In [ ]:
train_df = pd.read_csv("data/splits/train.csv")
val_df   = pd.read_csv("data/splits/val.csv")
test_df  = pd.read_csv("data/splits/test.csv")

train_vids = set(train_df["video_id"])
val_vids   = set(val_df["video_id"])
test_vids  = set(test_df["video_id"])

assert len(train_vids & val_vids) == 0,   "LEAK: train-val video overlap!"
assert len(train_vids & test_vids) == 0,  "LEAK: train-test video overlap!"
assert len(val_vids & test_vids) == 0,    "LEAK: val-test video overlap!"
print("[splits] No video ID leakage across splits. Integrity verified.")

---
## 3 · Cache Embeddings (Frozen Backbone)

Pre-compute per-video feature matrices `(T, D)` as `.npy` files using frozen DINOv2-S/14.

In [ ]:
%%time
!python scripts/cache_embeddings.py --backbone dinov2_vits14

# Verify embeddings
import numpy as np
emb_files = glob.glob("data/embeddings/dinov2_vits14/**/*.npy", recursive=True)
print(f"\n[embeddings] Cached {len(emb_files)} video embedding files.")
if emb_files:
    sample = np.load(emb_files[0])
    print(f"[embeddings] Sample shape: {sample.shape} (T x D = frames x embed_dim)")

---
## 4 · Sequential Training

Train all 6 model configurations sequentially. Each run:
- Saves `best_model.pt` (best val AUC) and `last_model.pt` to `outputs/{run_name}/`
- Copies `config.yaml` beside the checkpoint
- Logs per-epoch metrics to `history.csv` and TensorBoard

### 4.1 · Training Configuration

In [ ]:
# ==================================================================
#  TRAINING QUEUE — Fast-Track Settings for Remaining Models
#  [NOTE] ViT-Base and DeiT-Small are already trained and saved!
# ==================================================================
import yaml, os

# 1. Faster hyperparameters for remaining models (cuts training time by 3x):
FAST_SETTINGS = {
    "configs/dinov2_frozen.yaml":   {"epochs": 10, "batch_size": 64, "early_stopping_patience": 3},
    "configs/dinov2_partial.yaml":  {"epochs": 8,  "batch_size": 64, "early_stopping_patience": 3},
    "configs/efficientnet_b0.yaml": {"epochs": 8,  "batch_size": 64, "early_stopping_patience": 3},
}

for cfg_path, overrides in FAST_SETTINGS.items():
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f:
            cfg = yaml.safe_load(f)
        cfg["training"]["epochs"] = overrides["epochs"]
        cfg["training"]["batch_size"] = overrides["batch_size"]
        cfg["training"]["early_stopping_patience"] = overrides["early_stopping_patience"]
        with open(cfg_path, "w") as f:
            yaml.safe_dump(cfg, f)
        print(f"[speedup] Updated {cfg_path}: epochs={overrides['epochs']}, batch_size={overrides['batch_size']}")

# 2. Queue of models to train (ViT and DeiT commented out since they are done):
TRAINING_CONFIGS = [
    # COMPLETED:
    # ("configs/vit_b16_finetune.yaml",     "ViT-Base/16 Full Fine-Tune (Supervised ImageNet)"),
    # ("configs/deit_small_finetune.yaml",  "DeiT-Small/16 Full Fine-Tune (Distilled Supervised)"),

    # REMAINING:
    ("configs/dinov2_frozen.yaml",        "DINOv2-S/14 Frozen Linear Probe (Self-Supervised)"),
    ("configs/dinov2_partial.yaml",       "DINOv2-S/14 Partial Fine-Tune (Last K Blocks)"),
    ("configs/efficientnet_b0.yaml",      "EfficientNet-B0 Full Fine-Tune (CNN Baseline)"),
    ("configs/video_dinov2_cached.yaml",  "Temporal Head on Cached DINOv2 Embeddings"),
]

print(f"\nTraining queue: {len(TRAINING_CONFIGS)} runs remaining")
for i, (cfg, desc) in enumerate(TRAINING_CONFIGS, 1):
    print(f"  [{i}] {desc}")
    print(f"      Config: {cfg}")

### 4.2 · Execute Training Runs

In [ ]:
import time, glob, os, yaml

training_times = {}
total_start = time.time()

for idx, (config_path, description) in enumerate(TRAINING_CONFIGS, 1):
    with open(config_path, "r") as f:
        cfg_data = yaml.safe_load(f)
    run_name = cfg_data.get("experiment", {}).get("name", os.path.splitext(os.path.basename(config_path))[0])
    ckpt_path = f"outputs/{run_name}/best_model.pt"

    print("\n" + "=" * 70)
    print(f"  [{idx}/{len(TRAINING_CONFIGS)}] {description}")
    print(f"  Config: {config_path}")
    print("=" * 70)

    # ── SKIP IF ALREADY FINISHED ──────────────────────────────
    if os.path.exists(ckpt_path):
        size_mb = os.path.getsize(ckpt_path) / 1e6
        print(f"  --> [SKIPPED] Checkpoint already exists: {ckpt_path} ({size_mb:.1f} MB)")
        print(f"  --> Moving to next model without re-training!\n")
        continue

    run_start = time.time()
    exit_code = os.system(f"python scripts/train.py --config {config_path}")
    elapsed = time.time() - run_start
    training_times[description] = elapsed

    if exit_code != 0:
        print(f"\n  [WARNING] Training failed for {config_path} (exit code: {exit_code})")
        print(f"  Continuing to next configuration...\n")
    else:
        print(f"\n  [DONE] Completed in {elapsed / 60:.1f} minutes")

total_elapsed = time.time() - total_start

print("\n" + "=" * 70)
print("  Training Summary")
print("=" * 70)
for desc, t in training_times.items():
    print(f"  {desc:<55} {t/60:>6.1f} min")
print(f"  {'TOTAL':<55} {total_elapsed/60:>6.1f} min")

# List generated checkpoints
print("\n[checkpoints] All saved checkpoints:")
for pt in sorted(glob.glob("outputs/**/best_model.pt", recursive=True)):
    size_mb = os.path.getsize(pt) / 1e6
    print(f"  {pt}  ({size_mb:.1f} MB)")

In [ ]:
!zip -r /kaggle/working/deepfake_outputs.zip outputs/ -x "*.tfevents*"


---
## 5 · Evaluation

Run evaluation for each trained checkpoint on the held-out test set. Reports accuracy, precision, recall, F1, AUC-ROC, and saves confusion matrix and ROC curve plots.

In [ ]:
%%time

checkpoint_dirs = sorted(glob.glob("outputs/*/best_model.pt"))
print(f"[evaluate] Found {len(checkpoint_dirs)} checkpoints to evaluate.\n")

for ckpt_path in checkpoint_dirs:
    run_name = os.path.basename(os.path.dirname(ckpt_path))
    print(f"\n{'─' * 60}")
    print(f"  Evaluating: {run_name}")
    print(f"  Checkpoint: {ckpt_path}")
    print(f"{'─' * 60}")

    exit_code = os.system(
        f"python scripts/evaluate.py "
        f"--checkpoint {ckpt_path} "
        f"--split-csv data/splits/test.csv"
    )

    if exit_code != 0:
        print(f"  [WARNING] Evaluation failed for {run_name}")

### 5.1 · Cross-Dataset Evaluation (Optional)

If a Celeb-DF dataset is available, evaluate the best model on it to test generalization.

In [ ]:
# ── Cross-dataset: Train on FaceForensics++, test on Celeb-DF ──
# Skip this cell if KAGGLE_CELEBDF_DIR is not set or doesn't exist

CELEBDF_SPLITS = "data/splits_celebdf/test.csv"

if KAGGLE_CELEBDF_DIR and os.path.isdir(str(KAGGLE_CELEBDF_DIR)):
    # Build Celeb-DF splits if not already built
    if not os.path.exists(CELEBDF_SPLITS):
        print("[cross-dataset] Building Celeb-DF test split...")
        from src.data.make_splits import create_and_save_splits
        # Assuming Celeb-DF faces are already extracted
        celebdf_processed = os.path.join(str(KAGGLE_CELEBDF_DIR), "processed")
        if os.path.isdir(celebdf_processed):
            create_and_save_splits(
                data_dir=celebdf_processed,
                output_dir="data/splits_celebdf",
                train_ratio=0.0,
                val_ratio=0.0,
                test_ratio=1.0,
                seed=42,
            )

    if os.path.exists(CELEBDF_SPLITS):
        # Evaluate best-performing model on cross-dataset
        best_ckpt = "outputs/dinov2_frozen/best_model.pt"  # or whichever performed best
        if os.path.exists(best_ckpt):
            print(f"\n[cross-dataset] Evaluating {best_ckpt} on Celeb-DF...")
            !python scripts/evaluate.py --checkpoint {best_ckpt} --test_data {CELEBDF_SPLITS}
        else:
            print("[cross-dataset] Best checkpoint not found. Skipping.")
    else:
        print("[cross-dataset] Celeb-DF splits not available. Skipping.")
else:
    print("[cross-dataset] Celeb-DF dataset not attached. Skipping cross-dataset evaluation.")
    print("  To enable: set KAGGLE_CELEBDF_DIR in Cell 0.3 and attach the dataset.")

---
## 6 · Compare All Runs

In [ ]:
!python scripts/compare_runs.py

# Display the comparison table
comparison_csv = "outputs/comparison_results.csv"
if os.path.exists(comparison_csv):
    results_df = pd.read_csv(comparison_csv)
    print("\n" + "=" * 70)
    print("  Consolidated Results Table")
    print("=" * 70)
    from IPython.display import display, Markdown
    display(results_df)

    # Also display the Markdown version for copy-paste into reports
    md_path = "outputs/comparison_results.md"
    if os.path.exists(md_path):
        with open(md_path, "r") as f:
            display(Markdown(f.read()))
else:
    print("[compare] No comparison results generated. Check evaluation outputs.")

### 6.1 · Training History Curves

In [ ]:
import matplotlib.pyplot as plt

history_files = sorted(glob.glob("outputs/*/history.csv"))

if history_files:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = [("train_loss", "val_loss", "Loss"), ("train_acc", "val_acc", "Accuracy (%)"), ("train_auc", "val_auc", "AUC-ROC")]

    for ax, (train_col, val_col, title) in zip(axes, metrics):
        for hf in history_files:
            run_name = os.path.basename(os.path.dirname(hf))
            try:
                df = pd.read_csv(hf)
                if train_col in df.columns:
                    ax.plot(df["epoch"], df[val_col], label=run_name, linewidth=1.5)
            except Exception:
                continue
        ax.set_xlabel("Epoch", fontweight="bold")
        ax.set_ylabel(title, fontweight="bold")
        ax.set_title(f"Validation {title}", fontsize=12, fontweight="bold")
        ax.legend(fontsize=7, loc="best")
        ax.grid(True, alpha=0.3)

    plt.suptitle("Training History: All Backbone Architectures", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("outputs/training_history_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("[plot] Saved -> outputs/training_history_comparison.png")
else:
    print("[plot] No history.csv files found.")

---
## 7 · Package Outputs for Download

Zip the entire `outputs/` directory (checkpoints, metrics, plots, history CSVs) into a single archive at `/kaggle/working/` for download.

In [ ]:
%%time
import zipfile
from pathlib import Path

OUTPUT_ZIP = "/kaggle/working/deepfake_outputs.zip"
outputs_dir = Path("outputs")

if outputs_dir.exists():
    total_size = 0
    file_count = 0

    with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in sorted(outputs_dir.rglob("*")):
            if file_path.is_file():
                # Skip TensorBoard event files to reduce size
                if "events.out.tfevents" in file_path.name:
                    continue
                # Skip last_model.pt to save space (we keep best_model.pt)
                if file_path.name == "last_model.pt":
                    continue
                arcname = str(file_path)
                zf.write(file_path, arcname)
                total_size += file_path.stat().st_size
                file_count += 1

    zip_size = os.path.getsize(OUTPUT_ZIP) / 1e6
    print(f"\n[download] Archive created: {OUTPUT_ZIP}")
    print(f"[download] Files archived : {file_count}")
    print(f"[download] Original size  : {total_size / 1e6:.1f} MB")
    print(f"[download] Compressed size: {zip_size:.1f} MB")
    print(f"\n[download] Contents:")
    with zipfile.ZipFile(OUTPUT_ZIP, "r") as zf:
        for info in zf.infolist()[:30]:
            print(f"  {info.filename}  ({info.file_size / 1e3:.1f} KB)")
        if len(zf.infolist()) > 30:
            print(f"  ... and {len(zf.infolist()) - 30} more files")
else:
    print("[download] No outputs/ directory found. Did training complete?")

print("\n" + "=" * 70)
print("  Download the zip from the 'Output' tab on the right panel -->")
print("=" * 70)

---
## 8 · Final GPU Utilization & Session Summary

In [ ]:
print("=" * 70)
print("  Final Session Summary")
print("=" * 70)

# GPU memory summary
if torch.cuda.is_available():
    print(f"\n  GPU Memory Allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  GPU Memory Reserved  : {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"  GPU Peak Memory      : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

# Disk usage
outputs_size = sum(f.stat().st_size for f in Path("outputs").rglob("*") if f.is_file()) if Path("outputs").exists() else 0
data_size = sum(f.stat().st_size for f in Path("data").rglob("*") if f.is_file()) if Path("data").exists() else 0
print(f"\n  Disk Usage:")
print(f"    outputs/ : {outputs_size / 1e9:.2f} GB")
print(f"    data/    : {data_size / 1e9:.2f} GB")

# Checkpoints generated
best_checkpoints = sorted(glob.glob("outputs/*/best_model.pt"))
print(f"\n  Checkpoints Saved: {len(best_checkpoints)}")
for ckpt in best_checkpoints:
    run = os.path.basename(os.path.dirname(ckpt))
    sz = os.path.getsize(ckpt) / 1e6
    print(f"    {run:<30} {sz:>7.1f} MB")

# Final timestamp
print(f"\n  Completed at: {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
print("\n" + "=" * 70)
print("  Remaining Kaggle GPU Quota Notes:")
print("  - Check quota: https://www.kaggle.com/me/account")
print("  - T4 weekly budget: ~30 hours | P100 weekly budget: ~20 hours")
print("  - Quota resets weekly on your account anniversary date.")
print("=" * 70)